# 03. Exploratory Data Analysis (EDA)
**Objective:** Understand the cleaned dataset through visualisations — trends, distributions, and outliers.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)

df = pd.read_csv('../data/processed/cleaned_retail_data.csv.gz', compression='gzip')
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

---
## 1. Dataset Summary Statistics

In [ ]:
print("=== SHAPE ===")
print(df.shape)
print("\n=== DATA TYPES ===")
print(df.dtypes)
print("\n=== NUMERICAL SUMMARY ===")
df[['Quantity','UnitPrice','TotalRevenue','AvgOrderValue','Recency','Frequency','Monetary']].describe().round(2)

---
## 2. Revenue Distribution — Detecting Outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
df_active = df[df['IsCancelled']==False]
axes[0].hist(df_active['TotalRevenue'].clip(upper=500), bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Revenue per Line Item Distribution (clipped at £500)')
axes[0].set_xlabel('Revenue (£)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df_active['TotalRevenue'].clip(upper=500), vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'))
axes[1].set_title('Boxplot — Revenue per Line Item')
axes[1].set_xlabel('Revenue (£)')
plt.tight_layout()
plt.show()

---
## 3. Monthly Revenue Trend

In [ ]:
monthly = df[df['IsCancelled']==False].groupby(['InvoiceYearMonth'])['TotalRevenue'].sum().reset_index()
monthly = monthly.sort_values('InvoiceYearMonth')

plt.figure(figsize=(14,5))
plt.plot(monthly['InvoiceYearMonth'], monthly['TotalRevenue']/1000, marker='o', linewidth=2.5, color='#2563EB')
plt.fill_between(monthly['InvoiceYearMonth'], monthly['TotalRevenue']/1000, alpha=0.15, color='#2563EB')
plt.xticks(rotation=45, ha='right')
plt.title('Monthly Revenue Trend (£ thousands)')
plt.xlabel('Month')
plt.ylabel('Revenue (£K)')
plt.tight_layout()
plt.show()

---
## 4. Top 10 Countries by Revenue

In [ ]:
country_rev = (df[df['IsCancelled']==False]
               .groupby('Country')['TotalRevenue'].sum()
               .sort_values(ascending=False).head(10))

plt.figure(figsize=(12,5))
bars = plt.barh(country_rev.index[::-1], country_rev.values[::-1]/1000, color='#7C3AED')
plt.xlabel('Revenue (£K)')
plt.title('Top 10 Countries by Total Revenue')
for bar, val in zip(bars, country_rev.values[::-1]/1000):
    plt.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2, f'£{val:,.1f}K', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. UK vs International Revenue Split

In [ ]:
geo = df[df['IsCancelled']==False].groupby('IsUK')['TotalRevenue'].sum()
plt.figure(figsize=(6,5))
plt.pie(geo.values, labels=geo.index, autopct='%1.1f%%', colors=['#2563EB','#F59E0B'],
        startangle=140, wedgeprops=dict(edgecolor='white', linewidth=2))
plt.title('UK vs International Revenue Split')
plt.show()

---
## 6. Sales by Day of Week

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df[df['IsCancelled']==False].groupby('DayOfWeek')['TotalRevenue'].sum().reindex(dow_order)

plt.figure(figsize=(10,4))
bars = plt.bar(dow.index, dow.values/1000, color='#10B981', edgecolor='white')
plt.title('Total Revenue by Day of Week')
plt.ylabel('Revenue (£K)')
plt.xlabel('Day')
for bar in bars:
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'£{bar.get_height():.0f}K', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

---
## 7. Sales by Time of Day

In [ ]:
tod_order = ['Morning','Afternoon','Evening','Night']
tod = df[df['IsCancelled']==False].groupby('TimeOfDay')['TotalRevenue'].sum().reindex(tod_order)

plt.figure(figsize=(8,4))
plt.bar(tod.index, tod.values/1000, color=['#FBBF24','#F59E0B','#EF4444','#6366F1'], edgecolor='white')
plt.title('Revenue by Time of Day')
plt.ylabel('Revenue (£K)')
plt.xlabel('Time of Day')
plt.tight_layout()
plt.show()

---
## 8. Top 10 Best Selling Products (by Revenue)

In [ ]:
top_prod = (df[df['IsCancelled']==False]
            .groupby('Description')['TotalRevenue'].sum()
            .sort_values(ascending=False).head(10))

plt.figure(figsize=(12,5))
plt.barh(top_prod.index[::-1], top_prod.values[::-1]/1000, color='#EC4899')
plt.xlabel('Revenue (£K)')
plt.title('Top 10 Products by Revenue')
plt.tight_layout()
plt.show()

---
## 9. Customer Segment Distribution (RFM)

In [ ]:
seg = df[df['CustomerSegment']!='Guest'].drop_duplicates('CustomerID')['CustomerSegment'].value_counts()

plt.figure(figsize=(9,4))
bars = plt.bar(seg.index, seg.values, color=['#10B981','#2563EB','#F59E0B','#EF4444','#7C3AED','#EC4899','#6366F1'])
plt.title('Customer Segments (RFM Analysis)')
plt.ylabel('Number of Customers')
plt.xlabel('Segment')
for bar in bars:
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             str(bar.get_height()), ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Basket Size Distribution

In [ ]:
basket = df[df['IsCancelled']==False].drop_duplicates('InvoiceNo')['BasketSize'].value_counts()
order = ['Small (< £50)','Medium (£50-£200)','Large (> £200)']
basket = basket.reindex(order)

plt.figure(figsize=(7,4))
plt.pie(basket.values, labels=basket.index, autopct='%1.1f%%',
        colors=['#10B981','#2563EB','#7C3AED'], startangle=140,
        wedgeprops=dict(edgecolor='white',linewidth=2))
plt.title('Order Basket Size Distribution')
plt.show()

---
## 11. Cancellation Rate by Month

In [ ]:
cancel_monthly = df.groupby('InvoiceYearMonth').agg(
    total=('InvoiceNo','count'),
    cancelled=('IsCancelled','sum')
).reset_index()
cancel_monthly['CancelRate'] = (cancel_monthly['cancelled']/cancel_monthly['total']*100).round(2)
cancel_monthly = cancel_monthly.sort_values('InvoiceYearMonth')

plt.figure(figsize=(14,4))
plt.plot(cancel_monthly['InvoiceYearMonth'], cancel_monthly['CancelRate'], marker='o', color='#EF4444', linewidth=2)
plt.fill_between(cancel_monthly['InvoiceYearMonth'], cancel_monthly['CancelRate'], alpha=0.15, color='#EF4444')
plt.xticks(rotation=45, ha='right')
plt.title('Monthly Cancellation Rate (%)')
plt.ylabel('Cancellation Rate (%)')
plt.xlabel('Month')
plt.tight_layout()
plt.show()

---
## 12. Repeat vs One-Time Customers

In [ ]:
cust_type = df[df['CustomerID']!='Guest'].drop_duplicates('CustomerID')['IsRepeatCustomer'].value_counts()
plt.figure(figsize=(6,4))
plt.pie(cust_type.values, labels=cust_type.index, autopct='%1.1f%%',
        colors=['#2563EB','#F59E0B'], startangle=90,
        wedgeprops=dict(edgecolor='white',linewidth=2))
plt.title('Repeat vs One-Time Customers')
plt.show()